In [ ]:
!pip install scikit-learn pypdf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 3.4 MB/s eta 0:00:00


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import numpy as np, re

# -------- Mini-corpus interno sobre neumonía (puedes editarlo si quieres) --------
CORPUS = {
"intro_neumonia.txt": """
La neumonía adquirida en la comunidad (NAC) es una infección del parénquima pulmonar.
Síntomas frecuentes: fiebre, tos con o sin expectoración, disnea, dolor torácico pleurítico.
Signos de alarma: saturación de O2 < 90%, confusión, hipotensión sistólica < 90 mmHg,
frecuencia respiratoria > 30/min, cianosis, compromiso hemodinámico.
Factores de riesgo: edad avanzada, EPOC, cardiopatía, diabetes, alcoholismo, inmunosupresión.
""",
"diagnostico.txt": """
El diagnóstico de neumonía es clínico y se apoya en imagen.
La radiografía de tórax muestra consolidación o infiltrados; la oximetría evalúa hipoxemia.
En casos moderados-graves se solicitan hemocultivos, antígenos urinarios y gases arteriales.
CURB-65 ayuda a definir sitio de manejo: Confusión, Urea elevada, Respiratorio >=30,
Blood pressure baja y edad 65 o más. Puntajes altos se benefician de hospitalización.
""",
"tratamiento.txt": """
Tratamiento empírico ambulatorio habitual: amoxicilina a dosis altas o amoxicilina/ácido clavulánico.
Alternativas según resistencias: macrólidos (azitromicina/claritromicina) o doxiciclina.
Hospitalizado no UCI: betalactámico (ampicilina/sulbactam o ceftriaxona) más macrólido;
como alternativa, monoterapia con fluoroquinolona respiratoria. Duración típica 5–7 días
si mejoría clínica y sin complicaciones. Desescalar según cultivos y evolución.
""",
"complicaciones_prevencion.txt": """
Complicaciones: derrame parapneumónico, empiema, absceso pulmonar, sepsis, SDRA, insuficiencia respiratoria.
Prevención: vacunación antigripal y antineumocócica, higiene de manos, control de comorbilidades,
cesación tabáquica y adherencia al tratamiento. Educación al alta sobre signos de alarma y control.
"""
}

# -------- Troceo simple por oraciones para crear pasajes (chunks) --------
def dividir_en_chunks(texto, max_palabras=120):
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    chunks, actual, cuenta = [], [], 0
    for s in oraciones:
        w = len(s.split())
        if cuenta + w > max_palabras and actual:
            chunks.append(" ".join(actual)); actual=[]; cuenta=0
        actual.append(s); cuenta += w
    if actual: chunks.append(" ".join(actual))
    return [c for c in chunks if len(c.split()) >= 20]

# Construimos listas de texto y fuente
textos, fuentes = [], []
for nombre, contenido in CORPUS.items():
    for ch in dividir_en_chunks(contenido, max_palabras=110):
        textos.append(ch)
        fuentes.append(nombre)

# TF-IDF
vectorizador = TfidfVectorizer(max_features=12000, ngram_range=(1,2))
X = vectorizador.fit_transform(textos)

print(f"Índice listo: {len(textos)} fragmentos | vocabulario={len(vectorizador.vocabulary_)} términos")

Índice listo: 4 fragmentos | vocabulario=329 términos


In [ ]:
# --- Reemplazo de la función preguntar: responde solo lo relevante ---
import re, numpy as np

# Diccionario de "intención" simple por palabras clave
INTENT_MAP = {
    "sintomas": ["síntoma", "sintoma", "signo", "alarma", "clínic", "disnea", "tos", "fiebre", "pleur", "taquipnea", "hipox"],
    "tratamiento": ["tratamiento", "antibiót", "antibiot", "empírico", "empirico", "doxiciclina", "macrólido", "macrolido", "betalactám", "betalactam", "ceftriaxona", "amoxicilina"],
    "diagnostico": ["diagnóstico", "diagnostico", "radiografía", "rayos x", "imagen", "oximetr", "hemocultivo", "antígeno", "curb", "gasometr"],
    "complicaciones": ["complicación", "complicacion", "empiema", "derrame", "absceso", "sepsis", "sdra", "insuficiencia"],
    "prevencion": ["prevención", "prevencion", "vacun", "higiene", "cesación", "cesacion", "control de comorbilidades"]
}

# Palabras médicas para filtrar fuera de dominio
PALABRAS_DOMINIO = set(sum(INTENT_MAP.values(), [])) | set(["neumon", "pulmon", "respiratoria"])

def detectar_intencion(q: str):
    ql = q.lower()
    puntajes = {k: sum(1 for kw in kws if kw in ql) for k, kws in INTENT_MAP.items()}
    # intencion mas probable (o None si 0)
    best = max(puntajes, key=puntajes.get)
    return best if puntajes[best] > 0 else None

def oraciones(texto: str):
    return [s.strip() for s in re.split(r'(?<=[\.\?\!])\s+', texto) if len(s.split()) >= 5]

def preguntar(pregunta: str, k:int=4, umbral: float = 0.06, max_frases:int=5) -> str:
    # 1) Fuera de dominio
    if not any(p in pregunta.lower() for p in PALABRAS_DOMINIO):
        return "No disponible en el contexto."

    # 2) Recuperación TF-IDF
    v = vectorizador.transform([pregunta])
    sims = (X @ v.T).toarray().ravel()
    top_idx = np.argsort(-sims)[:k]
    if sims[top_idx[0]] < umbral:
        return "No disponible en el contexto."

    # 3) Extraer solo frases relevantes según intención + similitud
    intencion = detectar_intencion(pregunta)  # p.ej. "sintomas", "tratamiento", etc.
    frases_candidatas = []
    fuentes_usadas = []

    # vector de la pregunta para puntuar frases por coseno
    qv = v.toarray().ravel()
    for i in top_idx:
        texto = textos[i]
        fuente = fuentes[i]
        for s in oraciones(texto):
            sv = vectorizador.transform([s]).toarray().ravel()
            # coseno
            num = float((sv * qv).sum())
            den = (np.linalg.norm(sv) * np.linalg.norm(qv) + 1e-8)
            score = num / den
            # filtrar por intención (si existe) usando palabras clave
            pasa_intencion = True
            if intencion:
                pasa_intencion = any(kw in s.lower() for kw in INTENT_MAP[intencion])
            if pasa_intencion:
                frases_candidatas.append((score, s, fuente))

    if not frases_candidatas:
        return "No disponible en el contexto."

    # 4) Ordenar por relevancia y construir respuesta breve
    frases_candidatas.sort(key=lambda x: -x[0])
    seleccion = []
    fuentes_usadas = []
    for sc, s, f in frases_candidatas:
        if all(s not in t for t in seleccion):  # evitar repetidos
            seleccion.append(s)
            fuentes_usadas.append(f)
        if len(seleccion) >= max_frases:
            break

    respuesta = " ".join(seleccion)
    # 5) Citas compactas (únicas)
    refs = " | ".join(dict.fromkeys(fuentes_usadas))  # quitar duplicados conservando orden
    return (respuesta[:900].rsplit(" ",1)[0] + "..." if len(respuesta) > 900 else respuesta) + f"\n\nFuentes: {refs}"

In [ ]:
!pip install gradio --quiet
import gradio as gr

def ui_fn(q):
    return preguntar(q)

gr.Interface(
    fn=ui_fn,
    inputs=gr.Textbox(label="Pregunta sobre neumonía", placeholder="Ej: signos de alarma, tratamiento, diagnóstico..."),
    outputs="text",
    title="RAG experto en neumonía ",
    description="Responde basado en un mini-corpus interno. Si no hay evidencia suficiente, dirá: 'No disponible en el contexto.'"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a169479cfc9ca8cfb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -U sentence-transformers faiss-cpu transformers accelerate unidecode datasets --quiet --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 16.4 MB/s eta 0:00:00


In [ ]:
# Guardar los textos del diccionario CORPUS en la carpeta ./data/
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

for fname, content in CORPUS.items():
    (DATA_DIR / fname).write_text(content.strip(), encoding="utf-8")

print(" Archivos guardados en ./data:")
for f in DATA_DIR.glob("*.txt"):
    print(" -", f.name)


 Archivos guardados en ./data:
 - tratamiento.txt
 - complicaciones_prevencion.txt
 - diagnostico.txt
 - intro_neumonia.txt


In [ ]:
import os, re
from pathlib import Path
from unidecode import unidecode

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def read_txts(folder=DATA_DIR):
    docs = []
    for p in folder.glob("*.txt"):
        t = p.read_text(encoding="utf-8", errors="ignore")
        docs.append({"id": p.stem, "text": t})
    return docs

def clean(s):
    return unidecode(re.sub(r"\s+", " ", s).strip())

def chunk_text(text, max_chars=1200, overlap=150):
    text = clean(text)
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        dot = text.rfind(".", start, end)
        if dot != -1 and dot > start + 200:
            end = dot + 1
        chunks.append(text[start:end].strip())
        start = max(end - overlap, end)
    return [c for c in chunks if len(c) > 40]

docs = read_txts()
print(f"{len(docs)} documentos cargados.")


4 documentos cargados.


In [ ]:
corpus = []
for d in docs:
    for i, ch in enumerate(chunk_text(d["text"])):
        corpus.append({"doc_id": d["id"], "chunk_id": f'{d["id"]}_{i}', "text": ch})

print(f" Corpus listo: {len(corpus)} fragmentos.")
print("Ejemplo:", corpus[0]["chunk_id"], "→", corpus[0]["text"][:150])


 Corpus listo: 4 fragmentos.
Ejemplo: tratamiento_0 → Tratamiento empirico ambulatorio habitual: amoxicilina a dosis altas o amoxicilina/acido clavulanico. Alternativas segun resistencias: macrolidos (azi


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, torch, time

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_models = {
    "pm-MiniLM-L12-v2": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "e5-multilingual-small": "intfloat/multilingual-e5-small",
}

def build_index(model_ckpt, texts):
    model = SentenceTransformer(model_ckpt, device=device)
    embs = model.encode([c["text"] for c in texts],
                        batch_size=64, convert_to_numpy=True,
                        show_progress_bar=True, normalize_embeddings=True)
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)  # IP con vectores normalizados = cosine
    index.add(embs.astype(np.float32))
    return model, index

# construimos índices para todos
indexes = {}
for short, ckpt in embedding_models.items():
    print(f"Construyendo índice para {short} ...")
    emb_model, faiss_index = build_index(ckpt, corpus)
    indexes[short] = {"model": emb_model, "index": faiss_index}
print("Listo")


Construyendo índice para pm-MiniLM-L12-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Listo


In [ ]:
def retrieve(query, which="e5-multilingual-small", k=5):
    entry = indexes[which]
    q_emb = entry["model"].encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = entry["index"].search(q_emb.astype(np.float32), k)
    hits = []
    for rank, idx in enumerate(I[0]):
        hits.append({
            "rank": rank+1,
            "score": float(D[0][rank]),
            "chunk_id": corpus[idx]["chunk_id"],
            "text": corpus[idx]["text"][:250] + ("..." if len(corpus[idx]["text"])>250 else "")
        })
    return hits

# prueba rápida
for h in retrieve("¿Cuáles son síntomas típicos de la neumonía?", which="e5-multilingual-small", k=3):
    print(f"[{h['rank']}] score={h['score']:.3f}  id={h['chunk_id']}\n{h['text']}\n")


[1] score=0.898  id=intro_neumonia_0
La neumonia adquirida en la comunidad (NAC) es una infeccion del parenquima pulmonar. Sintomas frecuentes: fiebre, tos con o sin expectoracion, disnea, dolor toracico pleuritico. Signos de alarma: saturacion de O2 < 90%, confusion, hipotension sistol...

[2] score=0.882  id=diagnostico_0
El diagnostico de neumonia es clinico y se apoya en imagen. La radiografia de torax muestra consolidacion o infiltrados; la oximetria evalua hipoxemia. En casos moderados-graves se solicitan hemocultivos, antigenos urinarios y gases arteriales. CURB-...

[3] score=0.868  id=complicaciones_prevencion_0
Complicaciones: derrame parapneumonico, empiema, absceso pulmonar, sepsis, SDRA, insuficiencia respiratoria. Prevencion: vacunacion antigripal y antineumococica, higiene de manos, control de comorbilidades, cesacion tabaquica y adherencia al tratamie...



In [ ]:


from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch, re
import numpy as np

device = 0 if torch.cuda.is_available() else -1

# --- MODELOS GENERATIVOS (todos públicos) ---
gen_models = {
    "mt5-small": "google/mt5-small",
    "flan-t5-base": "google/flan-t5-base",
}

def make_generator(ckpt):
    tok = AutoTokenizer.from_pretrained(ckpt)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(ckpt)
    return pipeline("text2text-generation", model=mdl, tokenizer=tok, device=device)

gen_pipes = {name: make_generator(ckpt) for name, ckpt in gen_models.items()}


# --- Construcción de contexto ---
def build_context(hits, max_ctx_chars=800):  # reducido para no pasar el límite
    ctx = ""
    for h in hits:
        if len(ctx) + len(h["text"]) + 1 <= max_ctx_chars:
            ctx += "\n- " + h["text"]
        else:
            break
    return ctx.strip()


# --- Generador con truncation seguro ---
def gen_answer(q, hits, gen_pipe, max_new_tokens=96):
    ctx = build_context(hits)
    prompt = (
        "Tarea: responde en ESPAÑOL con UNA FRASE MÉDICA COMPLETA, "
        "usando SOLO el CONTEXTO; si no está, responde exactamente: No encontrado.\n\n"
        f"Pregunta: {q}\n"
        f"Contexto:{ctx}\n\n"
        "Respuesta:"
    )

    tok = gen_pipe.tokenizer
    model = gen_pipe.model

    # tokenización con truncation
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=480).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = tok.decode(outputs[0], skip_special_tokens=True)

    ans = out.split("Respuesta:")[-1].strip()
    ans = re.sub(r"<extra_id_\d+>", "", ans).strip()

    return {"answer": ans, "context_len": len(ctx)}


# --- Función principal RAG ---
def rag(question, emb="e5-multilingual-small", gen="flan-t5-base", k=2):
    hits = retrieve(question, which=emb, k=k)
    out = gen_answer(question, hits, gen_pipes[gen])
    return {
        "question": question,
        "answer": out["answer"],
        "embedding": emb,
        "generator": gen,
        "sources": [h["chunk_id"] for h in hits[:k]]
    }


# --- PRUEBAS RÁPIDAS ---
print(" Probando RAG...\n")

print(rag("¿Cuáles son los síntomas más frecuentes de la neumonía?"))
print(rag("¿Cuál es el tratamiento empírico inicial recomendado en NAC leve?"))


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


 Probando RAG...

{'question': '¿Cuáles son los síntomas más frecuentes de la neumonía?', 'answer': 'frecuentes: fiebre, tos con o sin expectoración, disnea, dolor toracico pleuritico', 'embedding': 'e5-multilingual-small', 'generator': 'flan-t5-base', 'sources': ['intro_neumonia_0', 'diagnostico_0']}
{'question': '¿Cuál es el tratamiento empírico inicial recomendado en NAC leve?', 'answer': 'Azitromicina/claritromicina', 'embedding': 'e5-multilingual-small', 'generator': 'flan-t5-base', 'sources': ['tratamiento_0', 'intro_neumonia_0']}


In [ ]:
tests = [
    "¿Cuáles son los síntomas más frecuentes de la neumonía?",
    "¿Qué hallazgo radiológico típico aparece en neumonía lobar?",
    "¿Tratamiento empírico recomendado en NAC leve en adulto sano?"
]

pairs = [
    ("pm-MiniLM-L12-v2", "flan-t5-base"),
    ("pm-MiniLM-L12-v2", "mt5-small"),
    ("e5-multilingual-small", "flan-t5-base"),
    ("e5-multilingual-small", "mt5-small"),
]

rows = []
for emb, gen in pairs:
    ok = 0
    for q in tests:
        out = rag(q, emb=emb, gen=gen)
        ok += 1 if out["answer"] and out["answer"].lower() != "no encontrado" else 0
    rows.append({"embedding": emb, "generador": gen, "respuestas_no_vacias": ok, "de": len(tests)})

import pandas as pd
pd.DataFrame(rows).sort_values(["respuestas_no_vacias"], ascending=False).reset_index(drop=True)


,embedding,generador,respuestas_no_vacias,de
0,pm-MiniLM-L12-v2,flan-t5-base,3,3
1,pm-MiniLM-L12-v2,mt5-small,3,3
2,e5-multilingual-small,flan-t5-base,3,3
3,e5-multilingual-small,mt5-small,3,3


In [ ]:
print(" Sistema RAG interactivo listo (escribe tu pregunta o ENTER para salir)\n")

while True:
    q = input("Pregunta: ").strip()
    if not q:
        print("Saliendo del modo interactivo.")
        break

    out = rag(q, emb="e5-multilingual-small", gen="flan-t5-base", k=2)
    print("\n Respuesta:", out["answer"])
    print(" Fragmentos usados:", out["sources"])
    print("-" * 80)


 Sistema RAG interactivo listo (escribe tu pregunta o ENTER para salir)


 Respuesta: frecuentes: fiebre, tos con o sin expectoración, disnea, dolor toracico pleuritico
 Fragmentos usados: ['intro_neumonia_0', 'diagnostico_0']
--------------------------------------------------------------------------------
